# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library. All data entities (record sets, fields, etc.) are referenced strictly by their `@id`, ensuring unambiguous identification.

### Dataset Source
The dataset source is provided via a Croissant schema URL (see below) using the [MLCommons Croissant](https://mlcommons.org/croissant/) metadata format.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
List available record sets, their `@id`, and fields using the Croissant schema. This gives an overview of the data structure and helps you select which entities to load and analyze.

> **Tip:** All references to record sets and fields should use their canonical `@id`s.

In [ ]:
# Get all record sets and their @id from the dataset
record_sets = [r for r in metadata.record_sets]
print(f"Found {len(record_sets)} record set(s):\n")
for idx, rs in enumerate(record_sets):
    print(f"{idx+1}. @id: {rs.id}")
    print(f"   name: {rs.name}")
    print(f"   description: {rs.description}")
    # List columns / fields for this record set
    if rs.fields:
        print("   Fields:")
        for field in rs.fields:
            print(f"     - @id: {field.id} | {field.name} [{field.data_type}]")
    print()

## 3. Data Extraction
Choose a record set of interest and load its content using its `@id`. All column references use their canonical field `@id` as discovered above.

We'll load *all* record sets for demonstration, building a DataFrame for each.

In [ ]:
# Prepare a list of record set @id's
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Use the mlcroissant Dataset.records method to load data
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set '{record_set_id}' with columns:")
    print(df.columns.tolist())
    print()

Let's display the first few records for the principal record set. We'll use the first available record set from the schema.

In [ ]:
# Specify which record set to work with (use the first one by default)
if len(record_set_ids) > 0:
    main_rs_id = record_set_ids[0]
    display(dataframes[main_rs_id].head())
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Select numeric fields for basic EDA: filtering, normalization, grouping, etc. All column references are by their column `@id`.

You may need to refer to the output above for available field `@id`s and their meaning, then adjust your analysis accordingly.

In [ ]:
# Choose a numeric field for analysis (Substitute with an actual @id from the columns)
df = dataframes.get(main_rs_id)
if df is not None and len(df) > 0:
    # Heuristically pick a numeric field
    numeric_candidate = None
    for col in df.columns:
        # Pandas attempts conversion to determine numerical fields
        # You should replace the below with an actual numeric field @id for robust analysis
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
            if np.issubdtype(df[col].dtype, np.number):
                numeric_candidate = col
                break
        except Exception:
            continue
    if numeric_candidate is not None:
        print(f"Using field '@id': {numeric_candidate} for numeric filtering.")

        # Filter for records above threshold
        threshold = df[numeric_candidate].mean() if df[numeric_candidate].mean() is not np.nan else 0.0
        filtered_df = df[df[numeric_candidate] > threshold].copy()
        print(f"Filtered records ({numeric_candidate} > {threshold:.2f}) count: {len(filtered_df)}")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_candidate}_normalized"] = (
            filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()
        ) / (filtered_df[numeric_candidate].std() if filtered_df[numeric_candidate].std() != 0 else 1)

        print(f"Normalized values for field '{numeric_candidate}':")
        display(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

        # Group by a likely categorical column (heuristically pick str/object type column)
        group_candidate = None
        for col in df.columns:
            if df[col].dtype == 'object' and col != numeric_candidate:
                group_candidate = col
                break
        if group_candidate:
            grouped_df = filtered_df.groupby(group_candidate)[numeric_candidate].mean().reset_index()
            print(f"Mean {numeric_candidate} grouped by {group_candidate} (@id).")
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric field found in main record set.")
else:
    print("No data loaded for the principal record set.")

## 5. Visualization
Visualize data distributions or relationships between fields from the dataset.

Here, we'll plot a histogram of the chosen numeric field and, if available, a boxplot by the top categorical group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and len(df) > 0 and numeric_candidate is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_candidate].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_candidate} (@id)")
    plt.xlabel(numeric_candidate)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    if group_candidate is not None:
        # Show boxplot by group
        top = df[group_candidate].value_counts().index[:5]
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df[df[group_candidate].isin(top)], x=group_candidate, y=numeric_candidate)
        plt.title(f"{numeric_candidate} distribution by {group_candidate} (top 5)")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we loaded, explored, and visualized the FAIR² mass spectrometry dataset using the `mlcroissant` library, referencing all entities (record sets, fields, columns) by their `@id`. You can extend this notebook for deeper domain-specific analysis, machine learning, or integration with additional Croissant-compliant datasets.